| 구분 | 추출 요약 (Extractive) | 생성 요약 (Abstractive) |
| :--- | :--- | :--- |
| **핵심 원리** | 원문에서 중요한 문장을 그대로 골라냄 | 원문의 의미를 파악해 새로운 문장을 만듦 |
| **비유** | 복사 및 붙여넣기 (Copy & Paste) | 독후감 쓰기 (Paraphrasing) |
| **장점** | 팩트 왜곡이 없고 문법적으로 정확함 | 글이 매끄럽고 자연스러우며 요약 효율이 높음 |
| **단점** | 문장 간 연결이 어색할 수 있음 | '환각(Hallucination)' 현상으로 틀린 정보를 낼 수 있음 |
| **주요 도구** | TextRank, LexRank, TF-IDF 등 | GPT, Gemini 등 LLM |

### 1. TF-IDF: 단어의 '희귀성'에 집중하는 방식

가장 직관적인 통계적 방식입니다. 모든 문장에서 흔하게 등장하는 단어(예: '있다', '것', '하다')는 무시하고, **특정 문장에서만 유독 많이 등장하는 핵심 키워드**를 찾습니다.

* **핵심 전략**: "귀한 단어가 많이 들어있는 문장이 곧 핵심 문장이다."
* **방식**: 단어마다 '중요도 점수'를 매깁니다. 그 점수들을 다 합쳐서 문장 점수를 내고, 가장 높은 점수를 받은 문장을 요약문으로 채택합니다.
* **장점**: 계산이 매우 빠르고 구현이 쉽습니다.

---

### 2. TextRank: 문장 간의 '관계'에 집중하는 방식

구글 검색 엔진의 원리를 활용한 방식입니다. 단어 빈도보다는 "어떤 문장이 다른 문장들과 의미적으로 잘 연결되어 있는가?"를 봅니다.

* **핵심 전략**: "많은 문장으로부터 '이 문장이 중요하다'고 추천(유사도 연결)을 받은 문장이 핵심 문장이다."
* **방식**: 문장들을 점(Node)으로 만들고, 비슷한 단어를 공유하는 문장끼리 선(Edge)으로 잇습니다. 선이 많이 모이는 '중심지' 같은 문장을 핵심으로 뽑습니다.
* **장점**: 문서 전체의 맥락을 파악하는 능력이 TF-IDF보다 뛰어납니다.

---

### 3. LSA (잠재 의미 분석): 단어 뒤의 '주제'를 찾는 방식

단순히 글자가 똑같은지 보는 게 아니라, 그 단어가 담고 있는 숨겨진 의미(Topic)를 수학적으로 추출합니다.

* **핵심 전략**: "글자는 달라도 같은 주제(예: 정치, 경제)를 다루는 문장들을 묶어서 가장 대표적인 것을 뽑자."
* **방식**: 문서 전체에서 반복되는 단어들의 패턴을 분석해 몇 개의 '주제 덩어리'를 만듭니다. 그 주제를 가장 잘 설명하는 문장을 각 덩어리에서 하나씩 가져옵니다.
* **장점**: 유의어(예: 자동차-차량)가 섞여 있어도 같은 의미로 인식할 수 있습니다.

---

### 4. LexRank: '중요한 문장과의 연결'을 중시하는 방식

TextRank와 비슷하지만, 연결의 '질'을 더 따집니다.

* **핵심 전략**: "그저 연결이 많은 게 중요한 게 아니라, '중요한 문장'과 연결된 문장이 진짜 핵심이다."
* **방식**: 문장 간의 유사도를 구한 뒤, 특정 기준(임계값)을 넘지 못하는 약한 연결은 과감히 끊어버립니다. 살아남은 강력한 연결 고리들 사이에서 가장 영향력 있는 문장을 찾습니다.
* **장점**: 소음(무의미한 문장)을 걸러내는 능력이 탁월하여 뉴스 요약에 많이 쓰입니다.

---

### 5. MMR (최대 한계 관련성): '중복 제거'에 집중하는 방식

이 방식은 앞선 알고리즘들과 결합하여 많이 쓰이며, 요약 결과가 **다양한 정보**를 담게 만듭니다.

* **핵심 전략**: "이미 뽑힌 핵심 문장과 너무 비슷한 내용은 두 번 뽑지 않는다."
* **방식**: 1순위 문장을 먼저 뽑은 후, 2순위 문장을 고를 때는 '중요도'는 높으면서도 '1순위 문장과 내용은 다른' 문장을 선택합니다.
* **장점**: 요약문이 똑같은 말을 반복하지 않고 풍부한 정보를 담게 됩니다.

In [1]:
import pandas as pd
import re

# 1. 파일 불러오기
df = pd.read_csv('ai_times1.csv', encoding="cp949")

def clean_text(text):
    if not isinstance(text, str):
        return text
    
    # '?' 기호를 공백(' ')으로 치환
    text = text.replace('?', ' ')
    
    # 연속된 공백(두 칸 이상)을 한 칸으로 줄임
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# 2. 전처리 적용
df['context'] = df['context'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)

# 3. 깨끗해진 데이터 저장
df.to_csv('ai_times_clean.csv', index=False, encoding='utf-8-sig')

print("데이터 정제가 완료되었습니다. 'ai_times_clean.csv' 파일을 확인해보세요.")

데이터 정제가 완료되었습니다. 'ai_times_clean.csv' 파일을 확인해보세요.


In [2]:
import pandas as pd
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

class SummaryMachine:
    def __init__(self):
        self.okt = Okt()
        self.vectorizer = TfidfVectorizer()

    def get_nouns(self, text):
        """명사 추출 및 전처리"""
        if not isinstance(text, str): return ""
        nouns = self.okt.nouns(text)
        return " ".join([n for n in nouns if len(n) > 1])

    def split_sentences(self, text):
        """문장 분리"""
        sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s for s in sentences if len(s.strip()) > 5]

    # --- 1. TF-IDF 기반 요약 ---
    def tfidf_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
            top_indices = scores.argsort()[-top_n:][::-1]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 2. TextRank 기반 요약 ---
    def textrank_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            sim_mat = (tfidf_matrix * tfidf_matrix.T).toarray()
            nx_graph = nx.from_numpy_array(sim_mat)
            scores = nx.pagerank(nx_graph)
            top_indices = sorted(scores, key=scores.get, reverse=True)[:top_n]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 3. LSA (잠재 의미 분석) 기반 요약 ---
    def lsa_summary(self, text, top_n=3):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            # 특이값 분해 (SVD)로 잠재 의미 추출
            svd = TruncatedSVD(n_components=1) # 가장 중요한 주제 1개 추출
            svd.fit(tfidf_matrix)
            # 각 문장이 해당 주제와 얼마나 관련있는지 확인
            scores = np.abs(svd.components_ @ tfidf_matrix.T.toarray()).flatten()
            top_indices = scores.argsort()[-top_n:][::-1]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 4. LexRank 기반 요약 ---
    def lexrank_summary(self, text, top_n=3, threshold=0.1):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            sim_mat = cosine_similarity(tfidf_matrix)
            # 특정 임계값 이하의 유사도는 연결 끊기
            sim_mat[sim_mat < threshold] = 0
            # 정규화하여 인접 행렬로 변환 후 랭킹 계산
            nx_graph = nx.from_numpy_array(sim_mat)
            scores = nx.pagerank(nx_graph)
            top_indices = sorted(scores, key=scores.get, reverse=True)[:top_n]
            top_indices.sort()
            return " ".join([sentences[i] for i in top_indices])
        except: return " ".join(sentences[:top_n])

    # --- 5. MMR (중복 제거 강조) 기반 요약 ---
    def mmr_summary(self, text, top_n=3, lambda_val=0.5):
        sentences = self.split_sentences(text)
        if len(sentences) <= top_n: return " ".join(sentences)
        sent_nouns = [self.get_nouns(s) for s in sentences]
        try:
            tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
            scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
            sim_mat = cosine_similarity(tfidf_matrix)
            
            summary_indices = [np.argmax(scores)] # 첫 번째 문장은 가장 높은 점수
            remaining_indices = [i for i in range(len(sentences)) if i not in summary_indices]
            
            while len(summary_indices) < top_n:
                mmr_values = []
                for i in remaining_indices:
                    # 중요도 가중치 - (이미 선택된 문장과의 최대 유사도 가중치)
                    relevance = scores[i]
                    redundancy = max([sim_mat[i][j] for j in summary_indices])
                    mmr_val = lambda_val * relevance - (1 - lambda_val) * redundancy
                    mmr_values.append((mmr_val, i))
                
                next_idx = max(mmr_values, key=lambda x: x[0])[1]
                summary_indices.append(next_idx)
                remaining_indices.remove(next_idx)
            
            summary_indices.sort()
            return " ".join([sentences[i] for i in summary_indices])
        except: return " ".join(sentences[:top_n])

# --- 메인 실행부 ---
def main():
    file_name = 'ai_times_clean.csv'
    df = pd.read_csv(file_name)
    summarizer = SummaryMachine()
    
    print(f"총 {len(df)}개 기사에 대해 5가지 알고리즘 요약을 시작합니다...")

    df['TF-IDF'] = df['context'].apply(lambda x: summarizer.tfidf_summary(x, 5))
    df['TextRank'] = df['context'].apply(lambda x: summarizer.textrank_summary(x, 5))
    df['LSA'] = df['context'].apply(lambda x: summarizer.lsa_summary(x, 5))
    df['LexRank'] = df['context'].apply(lambda x: summarizer.lexrank_summary(x, 5))
    df['MMR'] = df['context'].apply(lambda x: summarizer.mmr_summary(x, 5))

    output_file = 'ai_times_results_final.csv'
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"완료! 결과 저장: {output_file}")

if __name__ == "__main__":
    main()

총 5개 기사에 대해 5가지 알고리즘 요약을 시작합니다...
완료! 결과 저장: ai_times_results_final.csv


In [1]:
import pandas as pd
ai_times = pd.read_csv('ai_times_results_final copy.csv')
ai_times

,title,context,TF-IDF,TextRank,LSA,LexRank,MMR,gemini
0,"KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움”","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...",KAIST 연구팀은 물리 법칙과 분자 에너지를 스스로 이해하여 정밀한 분자 구조를 ...
1,"구글, 논문 일러스트 생성 AI ‘페이퍼바나나’ 공개...""환각 낮추고 미적 기준 올려""","AI가 참고 문헌 검색과 정리를 넘어, 학술 논문에 들어가는 이미지까지 처리하게 됐...",구글은 베이징대학교 연구진과 6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...,페이퍼바나나는 구글의 '나노바나나 프로' 등 최신 비전-언어 모델(VLM)과 이미지...,구글은 베이징대학교 연구진과 6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...,페이퍼바나나는 구글의 '나노바나나 프로' 등 최신 비전-언어 모델(VLM)과 이미지...,구글은 베이징대학교 연구진과 6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...,"이 시스템은 기획, 시각화, 비평 등 단계별 전문 에이전트가 협업하여 기존 AI의 ..."
2,29000TWh 시대…전력 시스템 전환의 열쇠는 무엇인가,국제에너지기구(IEA)가 6일 ‘Electricity 2026’ 행사를 통해 글로벌...,IEA는 2026년까지 세계 전력 소비가 사상 최고 수준인 약 29 000테라와트시...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,행사는 세계적으로 전력 수요가 빠르게 증가하고 있는 시점에 맞춰 전력 시스템의 효율...,IEA는 2026년까지 세계 전력 소비가 사상 최고 수준인 약 29 000테라와트시...,"IEA는 산업 전기화와 AI 데이터센터, 전기차 등의 확대로 인해 2026년 세계 ..."
3,산업용 RAG가 실패하는 이유 중 하나는 데이터 '청킹' 방식 때문,중공업 등 일부 산업 분야에서 검색 증강 생성(RAG)이 별 효과를 보지 못하는 결...,중공업 등 일부 산업 분야에서 검색 증강 생성(RAG)이 별 효과를 보지 못하는 결...,중공업 등 일부 산업 분야에서 검색 증강 생성(RAG)이 별 효과를 보지 못하는 결...,"이 때문에 ""RAG의 신뢰성을 향상하는 것은 더 큰 모델을 구입하는 것이 아니라, ...","전처리 과정에 문제가 있다""라고 밝혔다. 이 때문에 ""RAG의 신뢰성을 향상하는 것...",중공업 등 일부 산업 분야에서 검색 증강 생성(RAG)이 별 효과를 보지 못하는 결...,"기존 방식은 문서를 일정 글자 수로 나누면서 표나 이미지, 기술 수치 등의 맥락을 ..."
4,"프랑스 연구팀, AI 심리·정서 추적 프레임워크 ‘DefMoN’ 공개",프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,이번 프레임워크는 심리학 이론을 바탕으로 판단 과정을 추적하고 이를 재현한 합성 데...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,프랑스 라이언 리서치 연구소(RRI) 김상백 박사팀이 AI의 정서·심리 판단 근거를...,특히 심리학적 개념에서 벗어나는 '구성개념 이탈'이나 특정 단어로 정답을 유추하는 ...,이 프레임워크는 바이런트의 방어 기제와 플루칙의 정서 이론 등 심리학적 모델을 기반...


In [2]:
num = 0

In [3]:
print(f'기사 {num+1}')
print("제목:", ai_times.at[num,'title'])
print("원문:\n", ai_times.at[num,'context'].replace('. ', '.\n'))

기사 1
제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움”
원문:
 한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
특히, 분자구조의 에너지가 높을수록 ‘언덕’으로, 낮을수록 ‘골짜기’로 표현하는 등 알기 쉽게 지도 형태로 나타낸다고 전했다.
이를 통해 AI가 에너지가 가장 낮은 골짜기를 찾아 이동할 수 있도록 설계한 것이다.
이는 수학 이론인 리만 기하학을 적용한 것으로, AI가 화학 기본 원리인 “물질은 에너지가 가장 낮은 상태를 선호한다”라는 법칙을 학습한 결과다.
실험 결과, R-DM은 기존 AI보다 최대 20배 이상 높은 정확도를 보였다.
예측 오차는 정밀 양자역학 계산과 거의 차이가 없는 수준까지 줄었다는 설명이다.
이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
이 기술을 활용하면 신약 개발은 물론, 차세대 배터리 소재와 고성능 촉매 설계 등에 도움이 될 것이라는 설명이다.
많은 시간이 소요되던 분자 설계 과정을 크게 단축해 연구 개발 속도를 획기적으로 높여주는 ‘AI 시뮬레이터’로 작용한다고 전했다.
더불어, 화학 사고나 유해 물질 확산처럼 실험이 어려운 상황에서도 화학 반응 경로를 빠르게 예측할 수 있어서 환경 및 안전 분야에서 활용 가능성이 크다고 강조했다.
김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


### **프롬프트: 이 기사를 5문장으로 요약해줘 (제미나이 3.0 빠른모드)**

In [4]:
print(" [AI 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'gemini'].replace('. ', '.\n'))

 [AI 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

KAIST 연구팀은 물리 법칙과 분자 에너지를 스스로 이해하여 정밀한 분자 구조를 예측하는 AI 모델인 'R-DM'을 개발했습니다.
이 모델은 리만 기하학을 적용해 에너지가 가장 낮은 안정적인 상태를 찾아가는 방식으로, 단순한 형태 모방을 넘어 물리적 원리를 학습했습니다.
실험 결과 기존 AI보다 최대 20배 높은 정확도를 보였으며, 정밀 양자역학 계산에 비견되는 세계 최고 수준의 성능을 입증했습니다.
신약 개발, 차세대 배터리, 고성능 촉매 설계 등 다양한 분야에서 연구 시간을 획기적으로 단축하는 AI 시뮬레이터 역할을 할 것으로 기대됩니다.
김우연 교수는 이번 기술이 AI가 화학의 기본 원리를 이해한 첫 사례이며, 향후 신소재 개발 방식을 근본적으로 바꿀 것이라고 강조했습니다.


In [5]:
print(" [TF-IDF 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'TF-IDF'].replace('. ', '.\n'))

 [TF-IDF 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이는 수학 이론인 리만 기하학을 적용한 것으로, AI가 화학 기본 원리인 “물질은 에너지가 가장 낮은 상태를 선호한다”라는 법칙을 학습한 결과다.
많은 시간이 소요되던 분자 설계 과정을 크게 단축해 연구 개발 속도를 획기적으로 높여주는 ‘AI 시뮬레이터’로 작용한다고 전했다.
더불어, 화학 사고나 유해 물질 확산처럼 실험이 어려운 상황에서도 화학 반응 경로를 빠르게 예측할 수 있어서 환경 및 안전 분야에서 활용 가능성이 크다고 강조했다.
김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


In [9]:
print(" [MMR 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'MMR'].replace('. ', '.\n'))

 [MMR 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이는 수학 이론인 리만 기하학을 적용한 것으로, AI가 화학 기본 원리인 “물질은 에너지가 가장 낮은 상태를 선호한다”라는 법칙을 학습한 결과다.
많은 시간이 소요되던 분자 설계 과정을 크게 단축해 연구 개발 속도를 획기적으로 높여주는 ‘AI 시뮬레이터’로 작용한다고 전했다.
더불어, 화학 사고나 유해 물질 확산처럼 실험이 어려운 상황에서도 화학 반응 경로를 빠르게 예측할 수 있어서 환경 및 안전 분야에서 활용 가능성이 크다고 강조했다.
김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


In [6]:
print(" [TextRank 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'TextRank'].replace('. ', '.\n'))

 [TextRank 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


In [7]:
print(" [LSA 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'LSA'].replace('. ', '.\n'))

 [LSA 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
김우연 KAIST 교수는 “AI가 화학의 기본 원리를 이해하고 분자 안정성을 스스로 판단한 첫 사례”라며 “신소재 개발 방식을 근본적으로 바꿀 수 있는 기술”이라고 말했다.


In [8]:
print(" [LexRank 요약 결과] ")
print("원문 제목:", ai_times.at[num, 'title'],"\n")
print(ai_times.at[num, 'LexRank'].replace('. ', '.\n'))

 [LexRank 요약 결과] 
원문 제목: KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움” 

한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우하는 ‘물리법칙’을 스스로 이해, 구조까지 예측하는 AI 모델 ‘리만 확산 모델(R-DM)’을 개발했다고 10일 밝혔다.
이 모델은 분자의 에너지를 직접 고려한다는 점에서 차별적이라는 설명이다.
기존 AI가 분자의 모양을 단순히 흉내내는 차원이었다면, R-DM은 분자 내부에서 어떤 힘이 작용하는지를 고려해 스스로 구조를 다듬을 수 있다는 설명이다.
특히, 분자구조의 에너지가 높을수록 ‘언덕’으로, 낮을수록 ‘골짜기’로 표현하는 등 알기 쉽게 지도 형태로 나타낸다고 전했다.
이는 AI 분자 구조 예측 기술 중 세계 최고 수준 성능이라고 강조했다.
